# 00 · Preamble / Environment Setup

**Purpose:** one-time “power-on” cell that standardizes paths, config, randomness, plotting, and helper imports for the whole project. It keeps all other notebooks short and consistent.

### What this notebook does

* **Finds the project root** (walks up until it sees `configs/params.yaml`) and loads it.
* **Echoes key paths** (where the `.h5ad` is, where figures/tables will be saved) and **creates those folders** if missing.
* **Sets seeds & caps CPU threads** (reproducible results; no noisy oversubscription).
* **Sets Matplotlib defaults** and defines a tiny `savefig("figXX.png")` helper.
* **Adds** `notebooks/_lib` **to `sys.path`** and **smoke-tests** helper functions.
* **Loads & checks marker lists** (`S_mouse.txt`, `G2M_mouse.txt`) and prints counts.

### How to use it

1. Select the **`LimitCycle (mouse GSE154989)`** kernel for this notebook.
2. **Run all cells once per session** (or after changing `configs/params.yaml`).
3. Proceed to `01_ingest_build_anndata.ipynb` → `02_...` in order.

### Expected console prints

* `Repo root: .../gp1_limitcycle_mm_gse154989`
* `AnnData path: .../data/interim/mm_timecourse.h5ad`
* `[seed] set to 42` and thread caps dict
* `[helpers] loaded from .../notebooks/_lib`
* `[markers] S=…  G2M=…`

### Quick checks if something breaks

* Ensure `configs/params.yaml` exists and has:

  * `paths.anndata`, `paths.figures`, `paths.tables`
  * `markers.s_file`, `markers.g2m_file`
* Confirm marker files exist at those paths.
* If the root isn’t found, make sure you opened the notebook **inside** the project folder.


In [1]:
# --- Cell 1: paths + params (robust repo root + friendly checks) ---
from pathlib import Path
import yaml

def repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    # climb up to find configs/params.yaml
    for _ in range(8):
        if (p / "configs" / "params.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError(
        "Couldn't locate project root (configs/params.yaml). "
        "Open this notebook inside the project folder."
    )

BASE = repo_root()
PARAMS = BASE / "configs" / "params.yaml"

# load YAML safely
with open(PARAMS, "r") as fh:
    P = yaml.safe_load(fh) or {}

# provide sane defaults if keys are missing
P.setdefault("paths", {})
P["paths"].setdefault("anndata", "data/interim/mm_timecourse.h5ad")
P["paths"].setdefault("figures", "outputs/figures")
P["paths"].setdefault("tables",  "outputs/tables")

P.setdefault("markers", {})
P["markers"].setdefault("s_file",  "configs/markers/S_mouse.txt")
P["markers"].setdefault("g2m_file","configs/markers/G2M_mouse.txt")

P.setdefault("model", {})
P["model"].setdefault("seed", 42)

# unpack for convenience
PATHS = P["paths"]; MODEL = P["model"]; MARK = P["markers"]

# ensure output dirs exist
(fig_dir := BASE / PATHS["figures"]).mkdir(parents=True, exist_ok=True)
(tab_dir := BASE / PATHS["tables"]).mkdir(parents=True, exist_ok=True)

# echo what we’re using
print("Repo root      :", BASE)
print("Params file    :", PARAMS)
print("AnnData path   :", BASE / PATHS["anndata"])
print("Figures dir    :", fig_dir)
print("Tables dir     :", tab_dir)
print("S markers file :", BASE / MARK["s_file"])
print("G2M markers    :", BASE / MARK["g2m_file"])
print("Seed           :", MODEL["seed"])


Repo root      : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/gp1_limitcycle_mm_gse154989
Params file    : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/gp1_limitcycle_mm_gse154989/configs/params.yaml
AnnData path   : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/gp1_limitcycle_mm_gse154989/data/interim/mm_timecourse.h5ad
Figures dir    : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/gp1_limitcycle_mm_gse154989/outputs/figures
Tables dir     : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/gp1_limitcycle_mm_gse154989/outputs/tables
S markers file : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/gp1_limitcycle_mm_gse154989/configs/markers/S_mouse.txt
G2M markers    : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/gp1_limitcycle_mm_gse154989/configs/markers/G2M_mouse.txt
Seed           : 42


In [2]:
# --- Cell 2: seeds + thread caps (keep CPU libs calm/reproducible) ---
import os, random, numpy as np

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    print(f"[seed] set to {seed}")

def set_cpu_threads(n: int = 4, override: bool = False):
    envs = ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]
    for k in envs:
        if override or os.environ.get(k) is None:
            os.environ[k] = str(n)
    print("[threads]", {k: os.environ.get(k) for k in envs})

set_seed(int(MODEL.get("seed", 42)))
set_cpu_threads(4, override=False)


[seed] set to 42
[threads] {'OMP_NUM_THREADS': '4', 'OPENBLAS_NUM_THREADS': '4', 'MKL_NUM_THREADS': '4', 'NUMEXPR_NUM_THREADS': '4'}


In [3]:
# --- Cell 3: plotting defaults + save helper ---
import matplotlib.pyplot as plt

# modest, readable defaults
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.grid"] = False

FIGDIR = BASE / PATHS["figures"]

def savefig(name: str, *, tight: bool = True):
    """
    Save current Matplotlib figure into outputs/figures/.
    Usage: plt.plot(...); savefig("fig01_s_vs_g2m_circle.png")
    """
    out = FIGDIR / name
    if tight:
        plt.tight_layout()
    plt.savefig(out, bbox_inches="tight")
    print("[saved]", out)


In [4]:
# --- Cell 4: add notebooks/_lib to sys.path and smoke-test helpers ---
import sys
LIB = BASE / "notebooks" / "_lib"
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))

try:
    import cc_limitcycle as cc
    print("[helpers] loaded from", LIB)
    # optional: show available functions
    for name in ("fit_circle","angles_from_center","orient_phases","fit_periodic","annulus_disk_gmm","circular_w1","piecewise_warp"):
        print("  -", name, hasattr(cc, name))
except Exception as e:
    print("[helpers] not loaded:", e)


[helpers] loaded from /home/secondbook5/JHU_Bioinformatics/SystemsBiology/gp1_limitcycle_mm_gse154989/notebooks/_lib
  - fit_circle True
  - angles_from_center True
  - orient_phases True
  - fit_periodic True
  - annulus_disk_gmm True
  - circular_w1 False
  - piecewise_warp True


In [5]:
# --- Cell 5: verify marker files & define a tiny loader ---
from pathlib import Path

S_FILE   = BASE / MARK["s_file"]
G2M_FILE = BASE / MARK["g2m_file"]

def load_markers(path: Path) -> list[str]:
    return [ln.strip() for ln in open(path, "r") if ln.strip() and not ln.startswith("#")]

assert S_FILE.exists(),  f"Missing S markers file: {S_FILE}"
assert G2M_FILE.exists(),f"Missing G2M markers file: {G2M_FILE}"

S_genes   = load_markers(S_FILE)
G2M_genes = load_markers(G2M_FILE)
print(f"[markers] S={len(S_genes)}  G2M={len(G2M_genes)}")
print(" sample S:", S_genes[:5])
print(" sample G2M:", G2M_genes[:5])


[markers] S=0  G2M=0
 sample S: []
 sample G2M: []
